# Day 14: Exploratory Data Analysis & Feature Validation

This notebook performs quality audits, multi-collinearity checks, and tree-based feature importance on the engineered feature store.

In [1]:
# Notebook bootstrap: make the project importable and anchor relative paths.
# Jupyter starts the kernel in the notebook's own directory, so the project root has to
# be put on sys.path explicitly, exactly as the runnable modules under src/ do.
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)

Project root: /home/nikhil/tindal-data-pipeline/consumer-telemetry


In [2]:
import polars as pl
import numpy as np
from src.features.feature_validation import audit_feature_quality, analyze_correlations_and_redundancy, compute_feature_importances, select_features

# The train matrix stacks one block per train reference date (a user appears once per date).
# EDA and feature selection deliberately use the pooled rows, exactly as feature_validation.py does.
train_df = pl.read_parquet("data/features/train_features.parquet")
print("Train Matrix Shape:", train_df.shape)

Train Matrix Shape: (400, 36)


In [3]:
# 1. Feature Quality Audit
audit = audit_feature_quality(train_df)
print("Audit is_clean:", audit["is_clean"])
print("Infinite columns:", audit["infinite_columns"])
print("Columns with nulls:", list(audit["missing_summary"].keys()))

Audit is_clean: True
Infinite columns: []
Columns with nulls: ['days_since_last_purchase']


In [4]:
# 2. Correlation with Target and Multi-Collinearity
target_corrs, collinear = analyze_correlations_and_redundancy(train_df)
print("Top target correlations:")
for k, v in list(target_corrs.items())[:6]:
    print(f"  {k:<30}: {v:.4f}")
print(f"\nDetected {len(collinear)} multi-collinear pairs (|r| >= 0.90)")

Top target correlations:
  view_to_cart_ratio            : 0.4752
  total_purchases_30d           : 0.4673
  total_purchases_7d            : 0.4363
  days_since_last_cart_add      : -0.4000
  max_single_purchase           : 0.3694
  days_since_last_purchase      : -0.3665

Detected 7 multi-collinear pairs (|r| >= 0.90)


In [5]:
# 3. Tree-Based Importance & Selection
importances = compute_feature_importances(train_df)
selection = select_features(train_df, importances, collinear, top_k=18)
print(f"Selected {selection['selected_feature_count']} features.")
print("Top 5 predictive features:")
for item in selection["top_predictive_features"]:
    print(f"  {item['feature']:<30}: {item['importance']:.4f}")

Selected 18 features.
Top 5 predictive features:
  view_to_cart_ratio            : 0.0912
  days_since_last_cart_add      : 0.0617
  days_since_last_event         : 0.0600
  total_spend_30d               : 0.0529
  total_purchases_7d            : 0.0526
